In [1]:
# ══════════════════════════════════════════════════════════════
# 05_feature_engineering.ipynb
# Objetivo: construir df_model.csv listo para DML
# Flujo: Yapo limpio → cruce SII (agregado) → imputación → export
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
from rapidfuzz import process, fuzz

ROOT = Path("..")

df  = pd.read_csv(ROOT / "data/processed/yapo_clean.csv")
sii = pd.read_excel(ROOT / "data/external/datos_sii.xlsx")

print(f"Yapo bruto:  {df.shape}")
print(f"SII bruto:   {sii.shape}")
print(f"\nColumnas Yapo:\n{df.columns.tolist()}")

Yapo bruto:  (17470, 21)
SII bruto:   (81611, 19)

Columnas Yapo:
['id', 'titulo', 'precio_clp', 'anio', 'combustible', 'transmision', 'kilometraje', 'vendedor', 'region', 'destacado', 'url', 'fecha_scraping', 'fecha_publicacion', 'comuna', 'marca', 'modelo', 'empresa', 'direccion', 'tipo_vendedor', 'descripcion', 'edad_auto']


In [2]:
# ── Variable de tratamiento ───────────────────────────────────
df["es_automotora"] = (
    df["tipo_vendedor"]
    .fillna("particular").astype(str)
    .str.strip().str.lower()
    .ne("particular")
    .astype(int)
)

# ── Soporte común: excluir pre-2005 ──────────────────────────
# Justificación: <2005 tiene solo 1.9% automotoras → viola positivity
df = df[df["anio"] >= 2005].copy()

# ── Log-precio ───────────────────────────────────────────────
df["log_precio"] = np.log(df["precio_clp"])

print(f"Shape post-filtro: {df.shape}")
print(f"\nDistribución D:")
print(df["es_automotora"].value_counts())
print(df["es_automotora"].value_counts(normalize=True).round(3))

Shape post-filtro: (16708, 23)

Distribución D:
es_automotora
0    11462
1     5246
Name: count, dtype: int64
es_automotora
0    0.686
1    0.314
Name: proportion, dtype: float64


In [3]:
# ── Normalizar texto: minúsculas, sin tildes, sin caracteres raros
def normalizar_txt(x):
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

# Aplicar al SII
sii["marca_norm"]  = sii["Marca"].map(normalizar_txt)
sii["modelo_norm"] = sii["Modelo"].map(normalizar_txt)
sii["llave_sii"]   = (
    sii["marca_norm"] + " " +
    sii["modelo_norm"] + " " +
    sii["Año"].astype(str)
)

print("SII normalizado — muestra:")
print(sii[["Marca","Modelo","Año","llave_sii"]].head(4))

SII normalizado — muestra:
          Marca Modelo   Año               llave_sii
0  ASTON MARTIN   DB11  2018  aston martin db11 2018
1  ASTON MARTIN   DB11  2019  aston martin db11 2019
2  ASTON MARTIN   DB11  2020  aston martin db11 2020
3  ASTON MARTIN   DB11  2021  aston martin db11 2021


In [4]:
# ── PASO CLAVE: colapsar SII antes del merge ─────────────────
# Sin esto el merge genera producto cartesiano (~97k filas)
# Cada llave marca+modelo+año → 1 fila con mediana/moda de versiones

sii_agg = (
    sii.groupby("llave_sii")
    .agg(
        tipo_carroceria = ("Tipo",            lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        cilindrada_cc   = ("Cilindrada (CC)", "median"),
        potencia_hp     = ("Potencia (HP)",   "median"),
        traccion        = ("Tracción",         lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        combustible_sii = ("Combustible",      lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        transmision_sii = ("Transmisión",      lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        puertas         = ("Puertas",          "median"),
        n_versiones     = ("Código SII",       "nunique")
    )
    .reset_index()
)

print(f"SII agregado: {sii_agg.shape}  ← debe ser << 81.611")
print(f"Llaves únicas: {sii_agg['llave_sii'].nunique()}")
print(f"\nn_versiones promedio por llave: {sii_agg['n_versiones'].mean():.1f}")

SII agregado: (25862, 9)  ← debe ser << 81.611
Llaves únicas: 25862

n_versiones promedio por llave: 3.2


In [5]:
# ── Normalización base Yapo ───────────────────────────────────
df["marca_norm"]  = df["marca"].map(normalizar_txt)
df["modelo_norm"] = df["modelo"].map(normalizar_txt)

# ── Fix 1: colapsar alfanuméricos con espacio ─────────────────
# "l 200" → "l200" | "bt 50" → "bt50" | "cx 5" → "cx5"
def colapsar_alfanumerico(x):
    x = re.sub(r'([a-z]+)\s+(\d+)$', r'\1\2', x)
    x = re.sub(r'([a-z]{1,3})\s+(\d+)', r'\1\2', x)
    return x.strip()

df["modelo_norm"] = df["modelo_norm"].map(colapsar_alfanumerico)

# ── Fix 2: aliases Mazda ──────────────────────────────────────
# "3" → "mazda3" | "6" → "mazda6" | "2" → "mazda2"
mazda_alias = {
    "2": "mazda2", "3": "mazda3",
    "5": "mazda5", "6": "mazda6",
    "323": "323",  "626": "626", "929": "929",
}
mask_mazda = df["marca_norm"] == "mazda"
df.loc[mask_mazda, "modelo_norm"] = (
    df.loc[mask_mazda, "modelo_norm"]
    .map(lambda x: mazda_alias.get(x.strip(), x))
)

# ── Fix 3: truncar modelos Nissan con versión embebida ────────
# "navara 4x2 2 3 mt" → "navara" | "x trail 7p 4x4" → "x trail"
modelos_2tok_nissan = {
    "x trail", "np 300", "np300", "d 21", "d 22",
    "d 40", "nv 350", "grand livina"
}

def truncar_nissan(x):
    for m in modelos_2tok_nissan:
        if x.startswith(m):
            return m
    tokens = x.split()
    if len(tokens) > 2:
        if re.match(r'^\d', tokens[1]) or tokens[1] in {"4x2","4x4","dc","cab","d"}:
            return tokens[0]
    return x

mask_nissan = df["marca_norm"] == "nissan"
df.loc[mask_nissan, "modelo_norm"] = (
    df.loc[mask_nissan, "modelo_norm"]
    .map(truncar_nissan)
)

# ── Llave Yapo final ──────────────────────────────────────────
df["llave_yapo"] = (
    df["marca_norm"] + " " +
    df["modelo_norm"] + " " +
    df["anio"].astype(str)
)

print("Muestra llaves Yapo corregidas:")
print(df[["marca","modelo","anio","llave_yapo"]].sample(6, random_state=42))

Muestra llaves Yapo corregidas:
           marca    modelo  anio               llave_yapo
1018         Kia    Carens  2024          kia carens 2024
47       Hyundai    Tucson  2023      hyundai tucson 2023
5717       Mazda    Mazda6  2006        mazda mazda6 2006
7126       Mazda        --  2010              mazda  2010
10662    Peugeot      3008  2012        peugeot 3008 2012
13424  Chevrolet  Colorado  2021  chevrolet colorado 2021


In [6]:
choices = sii_agg["llave_sii"].dropna().unique().tolist()

def best_match(x, score_cutoff=85):
    m = process.extractOne(
        x, choices,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=score_cutoff
    )
    if m is None:
        return pd.Series([None, np.nan])
    return pd.Series([m[0], m[1]])

print("Ejecutando fuzzy match primera pasada...")
df[["llave_sii_match", "score_match"]] = (
    df["llave_yapo"].apply(best_match)
)

df["tiene_match"] = df["llave_sii_match"].notna().astype(int)

print(f"\nResultado primera pasada:")
print(f"Con match:  {df['tiene_match'].sum()} ({df['tiene_match'].mean()*100:.1f}%)")
print(f"Sin match:  {df['tiene_match'].eq(0).sum()}")
print(f"\nScore match:")
print(df["score_match"].describe())

Ejecutando fuzzy match primera pasada...

Resultado primera pasada:
Con match:  14315 (85.7%)
Sin match:  2393

Score match:
count    14315.000000
mean        99.129980
std          2.625156
min         85.000000
25%        100.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: score_match, dtype: float64


In [7]:
# ── Fix KIA: "kia motors" → "kia" en el SII ──────────────────
# Ya está resuelto si normalizar_txt corrió bien.
# Esta celda verifica y fuerza el re-match solo sobre sin-match

mask_sin = df["tiene_match"] == 0
print(f"Sin match antes de segunda pasada: {mask_sin.sum()}")
print("\nTop marcas sin match:")
print(df.loc[mask_sin, "marca"].value_counts().head(10))

# Segunda pasada con cutoff más permisivo (82)
def best_match_v2(x, score_cutoff=82):
    m = process.extractOne(
        x, choices,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=score_cutoff
    )
    if m is None:
        return pd.Series([None, np.nan])
    return pd.Series([m[0], m[1]])

df.loc[mask_sin, ["llave_sii_match", "score_match"]] = (
    df.loc[mask_sin, "llave_yapo"]
    .apply(best_match_v2)
    .values
)

df["tiene_match"] = df["llave_sii_match"].notna().astype(int)

print(f"\nResultado segunda pasada:")
print(f"Con match:  {df['tiene_match'].sum()} ({df['tiene_match'].mean()*100:.1f}%)")
print(f"Sin match:  {df['tiene_match'].eq(0).sum()}")

Sin match antes de segunda pasada: 2393

Top marcas sin match:
marca
Kia        1141
Otra        349
Peugeot      69
Dfsk         53
Geely        52
Nissan       48
Otros +      45
Ram          44
Bmw          43
Ds           42
Name: count, dtype: int64

Resultado segunda pasada:
Con match:  15046 (90.1%)
Sin match:  1662


In [8]:
# ── Test crítico: ¿el no-match es aleatorio? ─────────────────
print("% automotoras según tiene_match:")
print(df.groupby("tiene_match")["es_automotora"].mean().round(3))
print()
print("Log-precio según tiene_match:")
print(df.groupby("tiene_match")["log_precio"].mean().round(3))
print()
print("Top marcas sin match (irreducible):")
print(df.loc[df["tiene_match"]==0, "marca"].value_counts().head(12))

% automotoras según tiene_match:
tiene_match
0    0.503
1    0.293
Name: es_automotora, dtype: float64

Log-precio según tiene_match:
tiene_match
0    16.195
1    16.064
Name: log_precio, dtype: float64

Top marcas sin match (irreducible):
marca
Kia        597
Otra       326
Peugeot     69
Dfsk        53
Geely       48
Nissan      45
Ds          42
Ram         42
Otros +     42
Opel        38
Bmw         35
Ford        26
Name: count, dtype: int64


In [9]:
# ── Merge correcto: Yapo (1) ← → SII_agg (1) ────────────────
n_antes = len(df)

df = df.merge(
    sii_agg,
    left_on="llave_sii_match",
    right_on="llave_sii",
    how="left"
)

n_despues = len(df)

# Control de calidad: el merge NO debe multiplicar filas
assert n_antes == n_despues, (
    f"⚠️ Merge multiplicó filas: {n_antes} → {n_despues}. "
    f"Revisar duplicados en sii_agg."
)

print(f"✅ Merge uno-a-uno confirmado: {n_despues} filas")
print(f"\nCobertura variables SII post-merge:")
for col in ["tipo_carroceria","cilindrada_cc","potencia_hp","traccion","puertas"]:
    pct_ok = df[col].notna().mean() * 100
    print(f"  {col:20s}: {pct_ok:.1f}% completo")

✅ Merge uno-a-uno confirmado: 16708 filas

Cobertura variables SII post-merge:
  tipo_carroceria     : 90.1% completo
  cilindrada_cc       : 90.1% completo
  potencia_hp         : 43.0% completo
  traccion            : 60.9% completo
  puertas             : 89.9% completo


In [10]:
# ── Potencia (HP): mediana por marca + tipo + cilindrada ──────
df["potencia_hp"] = df.groupby(
    ["marca_norm","tipo_carroceria","cilindrada_cc"]
)["potencia_hp"].transform(lambda x: x.fillna(x.median()))

df["potencia_hp"] = df.groupby(
    ["marca_norm","tipo_carroceria"]
)["potencia_hp"].transform(lambda x: x.fillna(x.median()))

df["potencia_hp"] = df["potencia_hp"].fillna(df["potencia_hp"].median())

# ── Tracción: moda por marca + modelo ────────────────────────
def moda_safe(x):
    m = x.mode()
    return x.fillna(m.iloc[0] if not m.empty else "4x2 (2WD)")

df["traccion"] = df.groupby(
    ["marca_norm","modelo_norm"]
)["traccion"].transform(moda_safe)
df["traccion"] = df["traccion"].fillna("4x2 (2WD)")

# ── Tipo carrocería: moda por marca ──────────────────────────
df["tipo_carroceria"] = df.groupby(
    ["marca_norm"]
)["tipo_carroceria"].transform(moda_safe)
df["tipo_carroceria"] = df["tipo_carroceria"].fillna("Sedan")

# ── Puertas: moda por tipo carrocería ────────────────────────
df["puertas"] = df.groupby(
    ["tipo_carroceria"]
)["puertas"].transform(lambda x: x.fillna(x.median()))
df["puertas"] = df["puertas"].fillna(4.0)

# ── Cilindrada: mediana por marca + modelo ────────────────────
df["cilindrada_cc"] = df.groupby(
    ["marca_norm","modelo_norm"]
)["cilindrada_cc"].transform(lambda x: x.fillna(x.median()))
df["cilindrada_cc"] = df["cilindrada_cc"].fillna(df["cilindrada_cc"].median())

# ── Reporte final de nulos ────────────────────────────────────
print("Nulos residuales post-imputación:")
cols_check = ["tipo_carroceria","cilindrada_cc","potencia_hp","traccion","puertas"]
for col in cols_check:
    n = df[col].isna().sum()
    print(f"  {col:20s}: {n} nulos ({n/len(df)*100:.2f}%)")

Nulos residuales post-imputación:
  tipo_carroceria     : 0 nulos (0.00%)
  cilindrada_cc       : 0 nulos (0.00%)
  potencia_hp         : 0 nulos (0.00%)
  traccion            : 0 nulos (0.00%)
  puertas             : 0 nulos (0.00%)


In [11]:
# ── Señales textuales del aviso ───────────────────────────────
# Uso: robustez y mediación, NO backdoor set principal
desc = df["descripcion"].fillna("").str.lower()

df["tiene_garantia"] = desc.str.contains(
    r"garant|certificad", regex=True).astype(int)

df["unico_dueno"] = desc.str.contains(
    r"unico due[nñ]o|primer due[nñ]o", regex=True).astype(int)

df["al_dia"] = desc.str.contains(
    r"al d[ií]a|permiso al d[ií]a|patente al d[ií]a", regex=True).astype(int)

df["flag_impecable"] = desc.str.contains(
    r"impecable|como nuevo|sin rayones|sin choque", regex=True).astype(int)

print("Variables NLP (% con señal positiva):")
for col in ["tiene_garantia","unico_dueno","al_dia","flag_impecable"]:
    print(f"  {col:20s}: {df[col].mean()*100:.1f}%")

Variables NLP (% con señal positiva):
  tiene_garantia      : 3.9%
  unico_dueno         : 3.9%
  al_dia              : 38.2%
  flag_impecable      : 9.4%


In [12]:
# ── Flag para análisis de sensibilidad ───────────────────────
# muestra_restringida = solo obs con match confirmado (score ≥ 82)
df["match_confirmado"] = (
    df["score_match"].notna() & (df["score_match"] >= 82)
).astype(int)

print("Distribución match_confirmado:")
print(df["match_confirmado"].value_counts())
print()
print("% automotoras por flag:")
print(df.groupby("match_confirmado")["es_automotora"].mean().round(3))

# ── Columnas finales del modelo ───────────────────────────────
out_cols = [
    # IDs y metadata
    "id", "url", "fecha_scraping",
    # Outcome y tratamiento
    "precio_clp", "log_precio", "es_automotora",
    # Confounders Yapo
    "anio", "edad_auto", "kilometraje",
    "marca", "modelo", "region", "comuna",
    "combustible", "transmision",
    # Confounders SII (nuevos)
    "tipo_carroceria", "cilindrada_cc", "potencia_hp",
    "traccion", "puertas",
    # Señales NLP
    "tiene_garantia", "unico_dueno", "al_dia", "flag_impecable",
    # Metadata de calidad del cruce
    "score_match", "match_confirmado", "n_versiones",
    "llave_yapo", "llave_sii_match"
]

df_model = df[[c for c in out_cols if c in df.columns]].copy()

print(f"\ndf_model final: {df_model.shape}")
print(f"Columnas: {df_model.columns.tolist()}")

# ── Guardar ───────────────────────────────────────────────────
df_model.to_csv(ROOT / "data/processed/df_model.csv", index=False)
print("\n✅ df_model.csv guardado correctamente")

Distribución match_confirmado:
match_confirmado
1    15046
0     1662
Name: count, dtype: int64

% automotoras por flag:
match_confirmado
0    0.503
1    0.293
Name: es_automotora, dtype: float64

df_model final: (16708, 29)
Columnas: ['id', 'url', 'fecha_scraping', 'precio_clp', 'log_precio', 'es_automotora', 'anio', 'edad_auto', 'kilometraje', 'marca', 'modelo', 'region', 'comuna', 'combustible', 'transmision', 'tipo_carroceria', 'cilindrada_cc', 'potencia_hp', 'traccion', 'puertas', 'tiene_garantia', 'unico_dueno', 'al_dia', 'flag_impecable', 'score_match', 'match_confirmado', 'n_versiones', 'llave_yapo', 'llave_sii_match']



✅ df_model.csv guardado correctamente


In [13]:
# ── Reporte resumen del notebook ──────────────────────────────
print("═" * 50)
print("RESUMEN FINAL 05_feature_engineering.ipynb")
print("═" * 50)

verificar = pd.read_csv(ROOT / "data/processed/df_model.csv")

print(f"\nObservaciones:         {len(verificar):,}")
print(f"Variables:             {verificar.shape[1]}")
print(f"% con match SII:       {verificar['match_confirmado'].mean()*100:.1f}%")
print(f"% automotoras (D=1):   {verificar['es_automotora'].mean()*100:.1f}%")
print(f"Log-precio media:      {verificar['log_precio'].mean():.3f}")
print(f"Rango años:            {verificar['anio'].min()} – {verificar['anio'].max()}")
print(f"\nNulos residuales:")
print(verificar[["tipo_carroceria","cilindrada_cc","potencia_hp",
                  "traccion","puertas"]].isna().sum())
print("\n✅ Listo para 06_DML.ipynb")

══════════════════════════════════════════════════
RESUMEN FINAL 05_feature_engineering.ipynb
══════════════════════════════════════════════════

Observaciones:         16,708
Variables:             29
% con match SII:       90.1%
% automotoras (D=1):   31.4%
Log-precio media:      16.077
Rango años:            2005 – 2026

Nulos residuales:
tipo_carroceria    0
cilindrada_cc      0
potencia_hp        0
traccion           0
puertas            0
dtype: int64

✅ Listo para 06_DML.ipynb
